In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [ ]:
current = Path.cwd()

protected_areas_gdb = current / "../raw_data/protected_areas_database/PADUS4_1VectorAnalysis_PADUS_Only.gdb"
protected_areas_layer = "PADUS4_1VectorAnalysis_PADUS_Only_Simp_SingP"
county_boundaries_shp = current / "../raw_data/census_population_county_boundaries/tl_2023_us_county/tl_2023_us_county.shp"

protected_areas_gdb

# 1. Load dataset

In [ ]:
df = gpd.read_file(protected_areas_gdb, layer=protected_areas_layer)
print(df.shape)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

# 2. Check coordinate reference system

In [ ]:
df.crs

In [ ]:
df = df.to_crs(epsg=5070)
df.crs

# 3. Key columns

In [ ]:
df.columns

### `GAP_Sts`
Filter to GAP status 1 and 2 — areas where development is not permitted

In [ ]:
df["GAP_Sts"].value_counts()

In [ ]:
# GAP_Sts is stored as a string in this dataset
df = df[(df["GAP_Sts"] == "1") | (df["GAP_Sts"] == "2")]
print(df["GAP_Sts"].value_counts())
print(f"Current number of observations: {df.shape[0]}")

# 4. Spatial join with counties

In [ ]:
df_county = gpd.read_file(county_boundaries_shp)
print(f"Before conversion: {df_county.crs}")
if df_county.crs != df.crs:
    df_county = df_county.to_crs(df.crs)
print(f"After conversion: {df_county.crs}")
df_county.head()

In [ ]:
df_county = df_county[["GEOID", "NAMELSAD", "geometry"]]
df_county.columns = ["geo_id", "county_name", "geometry"]
df_county["county_area"] = df_county.area
df_county

In [ ]:
df_overlap = gpd.overlay(df, df_county, how="intersection", keep_geom_type=True)
print(df_overlap.shape)
print(df_overlap.crs)
df_overlap.head()

In [ ]:
df_overlap["overlap_area"] = df_overlap.area
df_overlap["pct_protected"] = df_overlap["overlap_area"] / df_overlap["county_area"]
df_overlap.sort_values(by="pct_protected", ascending=False).head()

In [ ]:
dfj = df_overlap.groupby(["geo_id", "county_name"]).agg(
    protected_count        = ("overlap_area", "count"),
    total_protected_area_m = ("overlap_area", "sum"),
    pct_protected          = ("pct_protected", "sum")
).reset_index()
dfj

In [ ]:
dfj.nlargest(10, "pct_protected")

In [ ]:
dfj.to_csv("../processed_data/protected_areas_by_county.csv", index=False)